# Modelado NLP - Subconjunto en Inglés (Baseline y Cascada)

Este notebook ejecuta la fase de preprocesado y modelado sobre el subconjunto de tickets en inglés. El objetivo es establecer un *baseline* sólido (TF-IDF + modelos tradicionales) y evaluar iterativamente el impacto de técnicas más complejas (Embeddings, SMOTE) en la clasificación de la tripleta (Queue -> Type -> Priority).

### Carga de la Capa Silver y Aislamiento del Idioma

Se extrae la matriz validada y se filtra exclusivamente el tráfico en inglés, garantizando que el modelo NLP aprenda patrones léxicos consistentes antes de abordar la porción en español.

In [10]:
import pandas as pd 

# Cargamos el dataset df_final_silver
df_en = pd.read_parquet ("../data/interim/df_final_silver.parquet")

# Eliminamos las filas que no sean en inglés.
df_en = df_en[df_en['language'] == 'en']

# Eliminamos la columna 'languaje'
df_en = df_en.drop(columns=['language'])

# Comprobamos las dimensiones del dataset
print (f"\nLas dimensiones del dataframe son {df_en.shape}\n")

# Estadisticas del dataframe
df_en.info()

# Eliminamos espacios en blanco en los extremos de la columna 'body'
df_en['body'] = df_en['body'].str.strip()

# Comprobamos cuantas columnas de 'body' quedan con longitud 0.
print (f"\n Columnas que quedan con longitud 0: {(df_en['body'].str.len() == 0).sum()}")

# Comprobamos cuántas celdas en la columna body son exactamente iguales (ignorando mayúsculas/minúsculas) 
# a términos como "na", "n/a", "nan", "null", o "none".
df_en['body'] = df_en['body'].str.lower()
texto_inutil = ('na', 'n/a', 'nan', 'null', 'none')
print ("\nComprobamos texto no usable en 'body':")
print (df_en['body'].isin(texto_inutil).sum())

# Hacemos recuento de frecuencias normalizado (en porcentaje) sobre la columna queue
print("\nRecuento de frecuencias normalizado (en porcentaje) para la columna 'queue':")
print (df_en['queue'].value_counts(normalize=True) * 100)



Las dimensiones del dataframe son (23112, 6)

<class 'pandas.DataFrame'>
Index: 23112 entries, 0 to 23861
Data columns (total 6 columns):
 #   Column    Non-Null Count  Dtype 
---  ------    --------------  ----- 
 0   subject   23112 non-null  object
 1   body      23112 non-null  object
 2   answer    23112 non-null  object
 3   type      23112 non-null  object
 4   queue     23112 non-null  object
 5   priority  23112 non-null  object
dtypes: object(6)
memory usage: 1.2+ MB

 Columnas que quedan con longitud 0: 0

Comprobamos texto no usable en 'body':
0

Recuento de frecuencias normalizado (en porcentaje) para la columna 'queue':
queue
Technical Support                  31.762721
Product Support                    20.197300
Customer Service                   16.411388
IT Support                         13.010557
Billing and Payments               11.002942
Service Outages and Maintenance     4.296469
Sales and Pre-Sales                 3.318622
Name: proportion, dtype: float64


# Comprobamos varios datos del dataset.